In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch, json, os, time, re
from tqdm import tqdm
import pandas as pd

## Load Model

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import json
import time

# Load the model
model_name = "microsoft/Phi-3-mini-4k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,  # Increase to give room for multiple properties
    do_sample=False,
    temperature=0.0
)

# Your lists
nouns = ["aardvark", "apple", "bicycle", "cat", "dog"]  # replace with full list
properties = [
    "size", "weight", "color", "speed", "strength", "agility", "edibility",
    "friendliness", "durability", "intelligence", "lifespan", "softness"
]

# Function to query model for plausible properties
def get_properties_for_noun(noun, properties_list):
    prompt = f"Given the noun '{noun}', list all plausible properties from this list: {properties_list}. Return as a Python list of strings."
    result = pipe(prompt)[0]['generated_text']
    # Evaluate safely to convert string output to Python list
    try:
        prop_list = eval(result)
        if isinstance(prop_list, list):
            # Only keep properties that exist in properties_list
            prop_list = [p for p in prop_list if p in properties_list]
            return prop_list
    except:
        pass
    return []

# Generate JSON mapping
noun_property_dict = {}
for noun in nouns:
    noun_property_dict[noun] = get_properties_for_noun(noun, properties)
    time.sleep(0.1)  # small delay to avoid overloading GPU

# Save to file
with open("nouns_properties.json", "w") as f:
    json.dump(noun_property_dict, f, indent=4)

print("JSON file created: nouns_properties.json")

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.13it/s]
Device set to use cuda


KeyboardInterrupt: 